[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session3/solutions/Lab5_Text_to_SQL_Solutions.ipynb)

# Lab 5: Text-to-SQL — Talk to Your Data (SOLUTIONS)
## CCI Session 3

**Duration:** 15 minutes

### Clinical Scenario
> The nursing director at KHCC wants to query patient vitals data without knowing SQL. You'll build a system where anyone can ask "How many patients had fever in the ICU this week?" and get an answer from the database.

### Objective
- Build the text-to-SQL pipeline: question → SQL → execute → answer
- Use few-shot examples to improve SQL generation
- Create a conversational interface with memory

---
### Setup

In [ ]:
# Setup — clone the repo and install packages
import os

if not os.path.exists('/content/CCI'):
    !git clone https://github.com/IyadSultan/CCI.git /content/CCI
os.chdir('/content/CCI/session3/solutions')

!pip install openai pandas -q

import json
import pandas as pd
import sqlite3
from openai import OpenAI
from google.colab import userdata

# Make sure your API key is stored in Colab Secrets (click the key icon in the left sidebar)
# under the name: OPENAI_API_KEY
api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

print(f"Working directory: {os.getcwd()}")
print("Setup complete!")

---
## Cell 1 — Rebuild the Database

Load `vista_vitals.csv` and `vista_patients.csv` from the `../data/` folder, then load both into SQLite.

In [ ]:
# === CELL 1: SETUP DATABASE ===
df_vitals = pd.read_csv('../data/vista_vitals.csv')
df_patients = pd.read_csv('../data/vista_patients.csv')
conn = sqlite3.connect('khcc_vitals.db')
df_vitals.to_sql('vista_vitals', conn, if_exists='replace', index=False)
df_patients.to_sql('vista_patients', conn, if_exists='replace', index=False)

# Verify both tables
result_vitals = pd.read_sql("SELECT COUNT(*) as total FROM vista_vitals", conn)
result_patients = pd.read_sql("SELECT COUNT(*) as total FROM vista_patients", conn)
print(f"Database ready with {result_vitals['total'][0]} vitals rows and {result_patients['total'][0]} patients rows")

---
## Cell 2 — Schema Description

Describe both tables and their JOIN relationship so the LLM can write correct SQL.

In [ ]:
# === CELL 2: SCHEMA DESCRIPTION ===
schema_description = """
Table: vista_vitals
Columns:
- NUMBER (integer): unique record ID
- MRN (text): encrypted patient medical record number  
- DATE_TIME_VITALS_TAKEN (text): when vitals were measured, format 'YYYY-MM-DDTHH:MM:SS.0000000'
- VITAL_TYPE (text): one of 'BLOOD PRESSURE', 'PULSE', 'PULSE OXIMETRY', 'RESPIRATION', 'TEMPERATURE'
- DATE_TIME_VITALS_ENTERED (text): when data was entered into the system
- HOSPITAL_LOCATION (text): ward name, e.g. 'KHCC-4C-HUSSAM AL-HARIRI', 'KHCC-ICU-CRITICAL CARE'
- ENTERED_BY (text): staff who recorded the vitals
- RATE (text): the vital sign value. For blood pressure it's 'systolic/diastolic' like '120/80'. For others it's a numeric string.

Table: vista_patients  
Columns:
- MRN (text): encrypted patient medical record number (JOIN key with vista_vitals)
- DATE_ESTABLISHED (text): when patient file was created
- DATE_OF_DEATH (text): '-' if alive, date string if deceased
- DOB (text): date of birth
- SEX (text): 'MALE' or 'FEMALE'
- NATIONALITY (text): 'JOR', 'PSE', 'IRQ', 'SYR', etc.
- PROVINCE (text): 'AMMAN', 'IRBID', 'ZARQA', etc.
- MARITAL_STATUS (text): 'MARRIED', 'SINGLE', 'WIDOWED', 'DIVORCED'
- NAME (text): Fernet-encrypted patient name
- PATIENT_TYPE (text): null, 'EARLY DETECTION', 'SCREENING', 'REFERRAL'

Relationship: vista_vitals.MRN = vista_patients.MRN (many vitals per patient)

Important notes:
- RATE is stored as TEXT, not a number. Use CAST(RATE AS FLOAT) for numeric comparisons (except blood pressure).
- Blood pressure RATE is in format 'systolic/diastolic' like '120/80'.
- Temperature is in Fahrenheit. Fever = temperature > 100.4.
- Normal pulse range: 60-100 bpm.
- Normal SpO2: >= 95%.
- Use JOINs when questions involve patient demographics (sex, nationality, province, age, etc.).
"""

# Get sample rows from both tables
sample_vitals = pd.read_sql("SELECT * FROM vista_vitals LIMIT 3", conn)
sample_patients = pd.read_sql("SELECT * FROM vista_patients LIMIT 3", conn)
print("Schema description created.")
print("\nSample vitals rows:")
print(sample_vitals.to_string())
print("\nSample patients rows:")
print(sample_patients.to_string())

---
## Cell 3 — Basic Text-to-SQL

In [ ]:
# === CELL 3: TEXT TO SQL ===

def text_to_sql(question):
    """Convert a natural language question to a SQL query."""
    system_prompt = f"""You are a SQL expert. Given the following database schema, 
generate a SQLite query to answer the user's question.

{schema_description}

Rules:
- Return ONLY the SQL query, nothing else. No explanation, no markdown, no code fences.
- Only generate SELECT queries. Never generate INSERT, UPDATE, DELETE, DROP, or ALTER.
- Use CAST(RATE AS FLOAT) when comparing numeric vital sign values (not for BLOOD PRESSURE).
- Use single quotes for string literals.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content.strip()
    # Clean up any markdown code fences if present
    sql = sql.replace('```sql', '').replace('```', '').strip()
    return sql

# Test
test_question = "How many total vital sign readings are in the database?"
sql = text_to_sql(test_question)
print(f"Question: {test_question}")
print(f"Generated SQL: {sql}")

---
## Cell 4 — Full Pipeline: Execute and Answer

In [ ]:
# === CELL 4: FULL PIPELINE ===

def ask_database(question):
    """Full text-to-SQL pipeline: question -> SQL -> execute -> natural language answer."""
    
    # Step 1: Generate SQL
    sql = text_to_sql(question)
    print(f"Generated SQL: {sql}")
    
    # Step 2: Safety check - only allow SELECT queries
    sql_upper = sql.strip().upper()
    if not sql_upper.startswith('SELECT'):
        return {
            'sql': sql,
            'answer': 'BLOCKED: Only SELECT queries are allowed for safety.',
            'data': None
        }
    
    # Check for dangerous keywords
    dangerous = ['DROP', 'DELETE', 'INSERT', 'UPDATE', 'ALTER', 'CREATE', 'TRUNCATE']
    for keyword in dangerous:
        if keyword in sql_upper:
            return {
                'sql': sql,
                'answer': f'BLOCKED: Query contains dangerous keyword: {keyword}',
                'data': None
            }
    
    # Step 3: Execute the SQL
    try:
        result_df = pd.read_sql(sql, conn)
    except Exception as e:
        return {
            'sql': sql,
            'answer': f'SQL execution error: {str(e)}',
            'data': None
        }
    
    # Step 4: Generate natural language answer
    result_str = result_df.to_string()
    
    answer_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful clinical data analyst. Given a question and SQL query results, provide a clear, concise natural language answer. Be specific with numbers."},
            {"role": "user", "content": f"Question: {question}\n\nSQL Query: {sql}\n\nResults:\n{result_str}"}
        ],
        temperature=0
    )
    
    answer = answer_response.choices[0].message.content
    
    return {
        'sql': sql,
        'answer': answer,
        'data': result_df
    }

# Test with clinical questions (single-table and JOIN-based)
questions = [
    "How many patients had fever in the ICU?",
    "Which ward has the most vital sign readings?",
    "What is the average pulse rate across all patients?",
    "How many male patients from Amman had high blood pressure?",
    "What's the average temperature for patients over 60?",
    "Show me vitals for Jordanian patients in the ICU"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    result = ask_database(q)
    print(f"SQL: {result['sql']}")
    print(f"Answer: {result['answer']}")

---
## Cell 5 — Few-Shot Text-to-SQL

Few-shot examples with JOIN queries dramatically improve multi-table SQL generation.

In [ ]:
# === CELL 5: FEW-SHOT TEXT-TO-SQL ===

def text_to_sql_fewshot(question):
    """Convert a natural language question to SQL using few-shot examples."""
    system_prompt = f"""You are a SQL expert. Given the following database schema, 
generate a SQLite query to answer the user's question.

{schema_description}

Rules:
- Return ONLY the SQL query, nothing else. No explanation, no markdown, no code fences.
- Only generate SELECT queries. Never generate INSERT, UPDATE, DELETE, DROP, or ALTER.
- Use CAST(RATE AS FLOAT) when comparing numeric vital sign values (not for BLOOD PRESSURE).
- Use single quotes for string literals.
- Use JOINs when questions involve patient demographics (sex, nationality, province, age, etc.).

Examples:

Question: How many unique patients are there?
SQL: SELECT COUNT(DISTINCT MRN) as unique_patients FROM vista_vitals

Question: Show me all fever readings
SQL: SELECT * FROM vista_vitals WHERE VITAL_TYPE = 'TEMPERATURE' AND CAST(RATE AS FLOAT) > 100.4

Question: Count readings per ward
SQL: SELECT HOSPITAL_LOCATION, COUNT(*) as count FROM vista_vitals GROUP BY HOSPITAL_LOCATION ORDER BY count DESC

Question: How many female patients had fever?
SQL: SELECT COUNT(DISTINCT v.MRN) FROM vista_vitals v JOIN vista_patients p ON v.MRN = p.MRN WHERE v.VITAL_TYPE = 'TEMPERATURE' AND CAST(v.RATE AS FLOAT) > 100.4 AND p.SEX = 'FEMALE'
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        temperature=0
    )

    sql = response.choices[0].message.content.strip()
    sql = sql.replace('```sql', '').replace('```', '').strip()
    return sql

# Compare with and without few-shot
test_q = "Find all patients in the ICU with abnormal pulse"
print(f"Question: {test_q}")
print(f"\nWithout few-shot: {text_to_sql(test_q)}")
print(f"\nWith few-shot:    {text_to_sql_fewshot(test_q)}")

# Test JOIN-based questions
join_questions = [
    "How many male patients from Amman had high blood pressure?",
    "What's the average temperature for patients over 60?",
    "Show me vitals for Jordanian patients in the ICU"
]

print("\n\n=== Testing JOIN-based questions ===")
for q in join_questions:
    print(f"\nQ: {q}")
    print(f"SQL: {text_to_sql_fewshot(q)}")

---
## Cell 6 — Conversational Text-to-SQL

In [ ]:
# === CELL 6: CONVERSATIONAL TEXT-TO-SQL ===

class ConversationalSQL:
    """Text-to-SQL with conversation memory."""
    
    def __init__(self, connection, schema):
        self.conn = connection
        self.schema = schema
        self.system_message = {
            "role": "system",
            "content": f"""You are a clinical data analyst assistant. You help users query a hospital vitals database.

{schema}

When the user asks a question:
1. Generate a SQL query to answer it.
2. Format your response as:
   SQL: <the sql query>
   
Rules:
- Only generate SELECT queries.
- Use CAST(RATE AS FLOAT) for numeric comparisons (except blood pressure).
- Use context from previous messages for follow-up questions.
- Put ONLY the SQL after "SQL:" with no markdown or code fences."""
        }
        self.messages = [self.system_message]
    
    def chat(self, user_input):
        """Process a user question and return an answer."""
        # Add user message
        self.messages.append({"role": "user", "content": user_input})
        
        # Get SQL from model
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=self.messages,
            temperature=0
        )
        
        assistant_msg = response.choices[0].message.content
        
        # Extract SQL from response
        sql = None
        if 'SQL:' in assistant_msg:
            sql = assistant_msg.split('SQL:')[1].strip()
            sql = sql.replace('```sql', '').replace('```', '').strip()
        
        if sql and sql.upper().startswith('SELECT'):
            # Execute the SQL
            try:
                result_df = pd.read_sql(sql, self.conn)
                result_str = result_df.to_string()
                
                # Get natural language answer
                self.messages.append({"role": "assistant", "content": assistant_msg})
                self.messages.append({
                    "role": "user",
                    "content": f"The query returned these results:\n{result_str}\n\nPlease provide a clear natural language summary."
                })
                
                answer_response = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=self.messages,
                    temperature=0
                )
                
                answer = answer_response.choices[0].message.content
                self.messages.append({"role": "assistant", "content": answer})
                
                return {'sql': sql, 'answer': answer, 'data': result_df}
                
            except Exception as e:
                error_msg = f"SQL error: {str(e)}"
                self.messages.append({"role": "assistant", "content": error_msg})
                return {'sql': sql, 'answer': error_msg, 'data': None}
        else:
            self.messages.append({"role": "assistant", "content": assistant_msg})
            return {'sql': sql, 'answer': assistant_msg, 'data': None}
    
    def reset(self):
        """Clear conversation history."""
        self.messages = [self.system_message]
        print("Conversation reset.")

# Test multi-turn conversation
chat_sql = ConversationalSQL(conn, schema_description)

multi_turn = [
    "How many patients had fever?",
    "Which wards were they in?",
    "Show me the most recent fever reading"
]

for question in multi_turn:
    print(f"\n{'='*60}")
    print(f"User: {question}")
    result = chat_sql.chat(question)
    print(f"SQL:  {result['sql']}")
    print(f"Answer: {result['answer']}")

---
## Cell 7 — Text-to-SQL with Tool Calling

In [ ]:
# === CELL 7: TEXT-TO-SQL WITH TOOL CALLING ===

# Define the tool
tools = [
    {
        "type": "function",
        "function": {
            "name": "run_sql_query",
            "description": "Execute a SQL SELECT query against the hospital vitals database and return the results.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": "The SQL SELECT query to execute."
                    }
                },
                "required": ["sql"]
            }
        }
    }
]

def run_sql_query(sql):
    """Execute a SQL query and return results as a string."""
    # Safety check
    sql_upper = sql.strip().upper()
    if not sql_upper.startswith('SELECT'):
        return "ERROR: Only SELECT queries are allowed."
    
    dangerous = ['DROP', 'DELETE', 'INSERT', 'UPDATE', 'ALTER', 'CREATE', 'TRUNCATE']
    for keyword in dangerous:
        if keyword in sql_upper:
            return f"ERROR: Query contains dangerous keyword: {keyword}"
    
    try:
        result_df = pd.read_sql(sql, conn)
        return result_df.to_string()
    except Exception as e:
        return f"ERROR: {str(e)}"


def ask_with_tools(question):
    """Ask a question using tool calling for SQL execution."""
    messages = [
        {
            "role": "system",
            "content": f"""You are a clinical data analyst. Use the run_sql_query tool to answer questions about the hospital database.

{schema_description}

Always use the tool to query the database. After getting results, provide a clear answer."""
        },
        {"role": "user", "content": question}
    ]
    
    # First call - model should request tool use
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        temperature=0
    )
    
    assistant_message = response.choices[0].message
    
    # Check if the model wants to use a tool
    if assistant_message.tool_calls:
        messages.append(assistant_message)
        
        for tool_call in assistant_message.tool_calls:
            if tool_call.function.name == "run_sql_query":
                args = json.loads(tool_call.function.arguments)
                sql = args['sql']
                print(f"Tool called with SQL: {sql}")
                
                # Execute the query
                result = run_sql_query(sql)
                
                # Add tool result to messages
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })
        
        # Second call - model provides final answer
        final_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0
        )
        
        return final_response.choices[0].message.content
    else:
        return assistant_message.content

# Test tool calling approach
print("=== Tool Calling Text-to-SQL ===")
questions = [
    "How many unique patients are in the database?",
    "Show me the average temperature by ward",
    "Which patients had both fever and tachycardia?"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    answer = ask_with_tools(q)
    print(f"A: {answer}")

---
## Stretch Challenge — Error Handling with Retry

In [ ]:
# === STRETCH CHALLENGE: ERROR HANDLING ===

def ask_database_with_retry(question, max_retries=3):
    """Text-to-SQL pipeline with automatic error correction."""
    
    messages = [
        {
            "role": "system",
            "content": f"""You are a SQL expert. Generate a SQLite SELECT query for the user's question.

{schema_description}

Return ONLY the SQL query, nothing else. No explanation, no markdown, no code fences.
Only generate SELECT queries."""
        },
        {"role": "user", "content": question}
    ]
    
    for attempt in range(max_retries):
        # Generate SQL
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0
        )
        
        sql = response.choices[0].message.content.strip()
        sql = sql.replace('```sql', '').replace('```', '').strip()
        
        print(f"  Attempt {attempt + 1}: {sql}")
        
        # Safety check
        if not sql.upper().startswith('SELECT'):
            return {'sql': sql, 'answer': 'BLOCKED: Only SELECT allowed.', 'data': None}
        
        # Try to execute
        try:
            result_df = pd.read_sql(sql, conn)
            
            # Generate natural language answer
            answer_response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "Provide a clear, concise answer based on the query results."},
                    {"role": "user", "content": f"Question: {question}\nSQL: {sql}\nResults:\n{result_df.to_string()}"}
                ],
                temperature=0
            )
            
            return {
                'sql': sql,
                'answer': answer_response.choices[0].message.content,
                'data': result_df,
                'attempts': attempt + 1
            }
            
        except Exception as e:
            error_msg = str(e)
            print(f"  Error: {error_msg}")
            
            # Send error back to model for correction
            messages.append({"role": "assistant", "content": sql})
            messages.append({
                "role": "user",
                "content": f"That SQL query failed with error: {error_msg}. Please fix it and return only the corrected SQL query."
            })
    
    return {
        'sql': sql,
        'answer': f'Failed after {max_retries} attempts.',
        'data': None,
        'attempts': max_retries
    }

# Test with a tricky question
print("Testing error handling with retry...")
result = ask_database_with_retry("What percentage of patients have abnormal vitals?")
print(f"\nFinal answer ({result.get('attempts', '?')} attempts): {result['answer']}")

---
### KHCC Connection

This is the foundation for the **natural language clinical data query system** being planned
for KHCC ward dashboards. Imagine a nurse asking "Show me all ICU patients with unstable
vitals in the last 4 hours" and getting an instant answer — no SQL knowledge required.
The text-to-SQL pipeline you built here is the core of that system.